# 📚 UPPA Smart Assistant — Notebook 01 (version 1.2)
## Acquisition, nettoyage et tableau de bord de couverture

**Mini-projet M2 · Industry 4.0 & Logistique / services**

Ce notebook découvre des liens publics, collecte des fiches dans un périmètre limité, nettoie le HTML, audite la couverture **de l'échantillon découvert** et produit un corpus traçable pour le Notebook 02.

> **Important :** le tableau de bord mesure l'exploration configurée, **pas le nombre exhaustif de pages disponibles sur le site de l'UPPA**. Les cas d'admission et de recherche doivent être vérifiés sur les pages d'origine.

### Compétences travaillées
1. Délimiter un périmètre d’exploration et respecter les règles d’accès à un site public.
2. Lire une page HTML et **distinguer contenu utile, navigation, tableaux et métadonnées**.
3. Construire un processus d’acquisition **répétable et traçable**.
4. Extraire du **Markdown structuré** et un **JSON commun** aux trois domaines.
5. Vérifier les résultats, identifier les doublons et **auditer les informations temporelles**.
6. Exporter un corpus propre pour une indexation ultérieure.

**Prérequis :** Python, dictionnaires, fonctions, requêtes HTTP et HTML élémentaire.

**Question de recherche pédagogique :** *Une conversion HTML→texte suffit-elle à produire un corpus RAG fiable, en particulier pour les calendriers d’admission et les fiches de formation ?*

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="font-family:system-ui;background:#f7f5ef;border:1px solid #dcd8cb;border-radius:16px;padding:18px;color:#263c34">
  <div style="font-family:Georgia,serif;font-size:21px;margin-bottom:15px">Du site Web au corpus RAG</div>
  <div style="display:flex;flex-wrap:wrap;gap:9px;align-items:center">
    <div style="flex:1;min-width:145px;padding:14px;border:1px solid #d7ded7;border-radius:12px;background:#eaf0eb"><b>Pages UPPA</b><div style="font-size:12px">Formations · Admissions · Recherche</div></div>
    <span style="font-weight:bold;color:#ae6346">→</span>
    <div style="flex:1;min-width:145px;padding:14px;border:1px solid #d7ded7;border-radius:12px;background:white"><b>Acquisition</b><div style="font-size:12px">URL · robots.txt · HTTP · HTML</div></div>
    <span style="font-weight:bold;color:#ae6346">→</span>
    <div style="flex:1;min-width:145px;padding:14px;border:1px solid #d7ded7;border-radius:12px;background:white"><b>Extraction</b><div style="font-size:12px">Titre · sections · tableaux · liens</div></div>
    <span style="font-weight:bold;color:#ae6346">→</span>
    <div style="flex:1;min-width:145px;padding:14px;border:1px solid #d7ded7;border-radius:12px;background:#f3e8db"><b>Corpus</b><div style="font-size:12px">Markdown · JSONL · provenance</div></div>
  </div><p style="margin:15px 0 0;font-size:12px;color:#64746a">Ici : acquisition et préparation. Dans le Notebook 02 : chunking → embeddings → Qdrant.</p>
</div>"""))

---
## Étape 0 — Installer l'environnement Colab

Nous utilisons d’abord **BeautifulSoup**, **Markdownify** et **Pandas** : l’étudiant peut inspecter chaque opération. **Crawl4AI** est une extension optionnelle, plus lourde à installer et à exécuter.

Les cellules ci-dessous s’exécutent dans un environnement Google Colab standard. Pour une exécution locale, utiliser un notebook Jupyter avec Python ≥ 3.10 et les mêmes bibliothèques.

In [ ]:
%pip -q install "requests>=2.31" "beautifulsoup4>=4.12" "lxml>=5" "markdownify>=0.13" "pandas>=2.0" "matplotlib>=3.7"

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from markdownify import markdownify as html_to_markdown
from IPython.display import display, Markdown
from collections import Counter
from datetime import datetime, timezone
import hashlib, io, json, re, time, zipfile, requests, pandas as pd
import matplotlib.pyplot as plt

BASE = Path('/content/UPPA_RAG' if Path('/content').exists() else './UPPA_RAG').resolve()
RAW_DIR = BASE / 'raw'; PROCESSED_DIR = BASE / 'processed'; EXPORT_DIR = BASE / 'exports'
for directory in [RAW_DIR, PROCESSED_DIR, EXPORT_DIR]: directory.mkdir(parents=True, exist_ok=True)
print('Répertoire de travail :', BASE)

## Étape 1 — Cartographier les sources officielles

Nous ne partons pas d'un robot qui suit tous les liens. Nous constituons une **liste initiale d’URL précises** et vérifiables. Le site de formation contient des pages de catalogue et des fiches ; le site d’admission précise les procédures ; le site de recherche présente les laboratoires et unités de recherche.

> **Exercice 1.** Pour les deux masters, repérer les éléments qui devront être conservés : titre du diplôme, campus, parcours, conditions d’accès, année universitaire, thématiques de recherche. Quels champs sont communs et lesquels sont propres à un domaine ?

In [ ]:
SEEDS = [
  {'domain':'formation','type':'catalogue','url':'https://formation.univ-pau.fr/fr/catalogue/toutes-les-formations.html'},
  {'domain':'formation','type':'formation','url':'https://formation.univ-pau.fr/fr/catalogue/master-XB/master-mention-informatique-L3Y98DNU.html'},
  {'domain':'formation','type':'recherche_catalogue','url':'https://formation.univ-pau.fr/fr/trouver-votre-formation.html'},
  {'domain':'admission','type':'portail','url':'https://formation.univ-pau.fr/fr/admission.html'},
  {'domain':'admission','type':'procedure','url':'https://formation.univ-pau.fr/fr/admission/candidater-a-l-universite/candidater-en-master.html'},
  {'domain':'recherche','type':'portail','url':'https://recherche.univ-pau.fr/fr/laboratoires.html'},
  {'domain':'recherche','type':'unites','url':'https://recherche.univ-pau.fr/fr/laboratoires/toutes-les-unites-de-recherche.html'},
]
print('Liste contrôlée :', len(SEEDS), 'URL')
display(pd.DataFrame(SEEDS))

### Le Web public n'est pas une base librement aspirante

Le notebook vérifie le protocole `https`, les deux hôtes autorisés, puis consulte **`robots.txt`**. S’il ne parvient pas à établir le droit d’accès (réponse non vérifiable, erreur réseau), **il n’essaie pas d’extraire la page**. Aucun contournement, aucun login et aucun service tiers de proxy ne sont utilisés.

Contraintes imposées : **au plus 7 URL initiales**, requêtes espacées, taille HTML maximale, journal de collecte et aucune donnée personnelle d’étudiant. Les conditions d'utilisation et de réutilisation doivent être contrôlées séparément ; `robots.txt` ne les remplace pas.

In [ ]:
# Paramètres transparents : modifier éventuellement la limite, pas les garde-fous.
USER_AGENT = 'UPPA-RAG-TeachingPrototype/0.1 (coursework, limited requests)'
ALLOWED_HOSTS = {'formation.univ-pau.fr', 'recherche.univ-pau.fr'}
MAX_SEED_PAGES = 7  # Pages initiales ; l’exploration guidée suit ensuite des liens.
PAUSE_SECONDS = 2.0
HTTP_TIMEOUT = 12
MAX_HTML_BYTES = 2_000_000
TRY_LIVE = True              # False : exécuter les exemples fictifs, sans réseau.
ROBOTS_CACHE = {}
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT, 'Accept': 'text/html'})


def authorized_host(url):
    parsed = urlparse(url)
    return parsed.scheme == 'https' and parsed.hostname in ALLOWED_HOSTS and not parsed.username


def robots_permission(url):
    """(autorisé, motif) ; refuse si robots.txt n'a pas pu être vérifié."""
    if not authorized_host(url):
        return False, 'URL / hôte non autorisé'
    root = f'https://{urlparse(url).hostname}'
    if root not in ROBOTS_CACHE:
        robots_url = root + '/robots.txt'
        try:
            r = SESSION.get(robots_url, timeout=HTTP_TIMEOUT, allow_redirects=False)
            if r.status_code != 200:
                ROBOTS_CACHE[root] = (None, f'robots.txt non vérifiable (HTTP {r.status_code})')
            elif len(r.content) > 1_000_000:
                ROBOTS_CACHE[root] = (None, 'robots.txt excessivement volumineux')
            else:
                rules = RobotFileParser()
                rules.parse(r.text.splitlines())
                ROBOTS_CACHE[root] = (rules, 'robots.txt vérifié')
        except requests.RequestException as exc:
            ROBOTS_CACHE[root] = (None, f'robots.txt inaccessible : {type(exc).__name__}')
    rules, status = ROBOTS_CACHE[root]
    if rules is None:
        return False, status
    return bool(rules.can_fetch(USER_AGENT, url)), status

assert not authorized_host('http://formation.univ-pau.fr/fr/accueil.html')
assert not authorized_host('https://exemple.invalid/formation')
assert authorized_host(SEEDS[0]['url'])
print('Tests de validation des hôtes : OK')

## Étape 2 — Acquisition HTTP contrôlée

Chaque réponse retenue doit être une **page HTML**. Le code conserve le document brut pour permettre une nouvelle extraction sans refaire de requête. Il enregistre aussi un statut : succès, refus par robots, échec réseau, format inattendu ou redirection à vérifier.

> **Pourquoi les redirections sont-elles bloquées ici ?** Pour empêcher qu’une URL autorisée conduise automatiquement vers un autre hôte. Les URL redirigées doivent être examinées explicitement avant ajout à la liste initiale.

In [ ]:
def get_html_guarded(url):
    ok, msg = robots_permission(url)
    if not ok:
        return None, {'status':'skipped_robots', 'detail':msg, 'url':url}
    try:
        with SESSION.get(url, timeout=HTTP_TIMEOUT, stream=True, allow_redirects=False) as r:
            if r.status_code in (301,302,303,307,308):
                return None, {'status':'redirect_needs_review','detail':r.headers.get('Location'), 'url':url}
            if r.status_code != 200:
                return None, {'status':f'http_{r.status_code}','detail':'HTTP non-200','url':url}
            mime = r.headers.get('Content-Type','').lower()
            if 'text/html' not in mime:
                return None, {'status':'not_html','detail':mime,'url':url}
            received = bytearray()
            for part in r.iter_content(chunk_size=32_768):
                received.extend(part)
                if len(received) > MAX_HTML_BYTES:
                    return None, {'status':'too_large','detail':f'>{MAX_HTML_BYTES} octets','url':url}
            enc = r.encoding or 'utf-8'
            html = bytes(received).decode(enc, errors='replace')
            return html, {'status':'ok','detail':f'{len(received)} octets','url':url,
                          'content_type':mime}
    except requests.RequestException as exc:
        return None, {'status':'network_error','detail':type(exc).__name__,'url':url}


raw_live = []
log = []
if TRY_LIVE:
    for seed in SEEDS[:MAX_SEED_PAGES]:
        html, status = get_html_guarded(seed['url'])
        log.append({**seed, **status})
        print(f"{seed['domain']:9s} | {status['status']:22s} | {seed['url'][:68]}")
        if html:
            raw_live.append({**seed, 'html':html, 'origin':'live', 'source_url':seed['url']})
            filename = hashlib.sha256(seed['url'].encode()).hexdigest()[:16] + '.html'
            (RAW_DIR / filename).write_text(html, encoding='utf-8')
            time.sleep(PAUSE_SECONDS)
else:
    print('Mode sans réseau, pas de requête envoyée.')

display(pd.DataFrame(log)) if log else None
print('Pages publiques effectivement récupérées :', len(raw_live))

### Étape 2A — Découvrir les liens : pourquoi les « + » ne sont pas forcément un obstacle ?

Le **catalogue UPPA** comporte des rubriques repliées (Master, Licence, etc.). Dans la version examinée, leurs liens sont **déjà présents dans le HTML** : `BeautifulSoup` peut donc les extraire sans cliquer.

L’erreur d’une collecte limitée à quelques documents vient souvent du fait que le programme **liste** les liens, mais ne **visite** pas les pages correspondantes.

**Activité** : observer le nombre de liens du catalogue, puis les répartir par domaine. Nous allons ensuite activer un parcours contrôlé à deux niveaux, dans les limites définies par l’étudiant.

> Si un autre site injecte réellement les liens avec JavaScript après un clic, il faudra une exploration par navigateur (Playwright ou Crawl4AI). Ce n’est pas notre hypothèse par défaut pour le catalogue UPPA.

In [ ]:
def candidate_urls(html, base_url, max_links=12):
    soup = BeautifulSoup(html, 'lxml')
    found=[]
    for a in soup.select('a[href]'):
        url = urldefrag(urljoin(base_url, a['href']))[0]
        if authorized_host(url) and url not in found:
            found.append(url)
    return found[:max_links]

if raw_live:
    for link in candidate_urls(raw_live[0]['html'], raw_live[0]['source_url']):
        print('•',link)
else:
    print('Aucun lien officiel inspectable ici : les démonstrations HTML seront utilisées ci-dessous.')

## Étape 2B — Exploration contrôlée des pages liées (version améliorée)

**Objectif.** Dépasser les 6 à 7 URL initiales en suivant les liens des fiches de formation, procédures d’admission et pages des laboratoires, sans aspirer sans limites tout le domaine.

### Les deux niveaux de navigation

```text
Catalogue / Admission / Laboratoires  (niveau 0)
    ├─ Mention Master / procédure / unité  (niveau 1)
    └─ Parcours / précision / fiche liée   (niveau 2)
```

**Pourquoi ce choix ?** Le catalogue UPPA expose de nombreuses formations et parcours dans des liens HTML, même quand les rubriques sont visuellement repliées. L’intérêt pédagogique n’est pas de cliquer mécaniquement mais d’**extraire les liens, les qualifier, visiter les pages et conserver leur provenance**.

### Garde-fous

- Seulement les deux hôtes UPPA déjà autorisés ; uniquement `https`, pages HTML officielles et chemins ciblés.
- Vérification préalable de `robots.txt` pour chaque URL ; refus en cas d’incertitude.
- Pause entre les requêtes, plafonds **par domaine** et profondeur maximale de 2.
- Journal de collecte, codes d’erreur, déduplication des URL et métadonnées de parenté.
- Les sites externes des laboratoires ne sont pas visités automatiquement : ils demandent un audit de périmètre distinct.
- Le réglage `FORMATION_SCOPE="masters"` collecte un **échantillon de mentions et de parcours de master** ; il ne prétend pas couvrir tout le catalogue.

**Consigne.** Lire les réglages avant exécution. N’augmenter les limites qu’après accord sur le périmètre pédagogique et vérification des conditions d’accès.

In [ ]:
from collections import deque
from urllib.parse import urlsplit, urlunsplit

# ── Réglages du parcours ───────────────────────────────────────────────────────
EXPLORE_LINKS = True        # False = conserver seulement les pages SEEDS
FORMATION_SCOPE = 'masters' # 'masters' ou 'catalogue' (échantillon plafonné)
MAX_DEPTH = 2              # distance depuis une URL SEED
FOLLOW_LIMITS = {'formation': 20, 'admission': 10, 'recherche': 10}
ATTEMPT_LIMITS = {d: 2 * n for d, n in FOLLOW_LIMITS.items()}

# Remarque : les limites ci-dessus concernent les pages supplémentaires réussies.
# Les redirections et erreurs sont consignées mais ne donnent jamais lieu à des relances illimitées.

def normalize_internal_url(href, base_url):
    if not href or href.startswith(('mailto:', 'tel:', 'javascript:', '#')):
        return None
    u = urlsplit(urljoin(base_url, href))
    clean = urlunsplit((u.scheme, u.netloc, u.path, '', ''))
    if not authorized_host(clean) or not u.path.endswith('.html'):
        return None
    return clean


def classify_discovered_url(url):
    path = urlsplit(url).path
    if path.startswith('/fr/catalogue/') and path.endswith('.html'):
        # Mode échantillonné « masters » : les mentions ET leurs parcours.
        if FORMATION_SCOPE == 'masters' and '/master-XB/' not in path:
            return None
        return 'formation'
    if path.startswith('/fr/admission/') and path.endswith('.html'):
        return 'admission'
    if path.startswith('/fr/laboratoires/') and path.endswith('.html'):
        return 'recherche'
    return None


def discover_document_links(record):
    """Liens de la page, y compris les panneaux repliés dans le HTML."""
    origin = record['source_url']
    soup = BeautifulSoup(record['html'], 'lxml')
    root = soup.select_one('main, [role="main"], article, #contenu, #content') or soup.body or soup
    discovered = []
    local_seen = set()
    for a in root.select('a[href]'):
        target = normalize_internal_url(a.get('href'), origin)
        if target is None or target in local_seen:
            continue
        domain = classify_discovered_url(target)
        if not domain:
            continue
        local_seen.add(target)
        discovered.append({
            'url': target, 'domain': domain, 'type': 'fiche_liee',
            'link_label': a.get_text(' ', strip=True)[:150] or None,
            'discovered_from': origin,
            'crawl_depth': record.get('crawl_depth', 0) + 1
        })
    # Les fiches Informatique/Industry 4.0/SIGLIS sont prioritaires dans l'échantillon.
    priority_high = ('master-mention-informatique', 'industry-4-0', 'logistique-industrielle')
    priority_mid = ('master-mention-genie-industriel', 'master-mention-mathematiques',
                    'master-mention-management-des-systemes', 'master-mention-management-et-commerce',
                    'master-mention-economie-appliquee')
    def thematic_priority(item):
        u = item['url']
        return 0 if any(x in u for x in priority_high) else (
               1 if any(x in u for x in priority_mid) else 2)
    discovered.sort(key=lambda it: (thematic_priority(it), it['url']))
    return discovered


def crawl_controlled(start_pages):
    queues = {domain:deque() for domain in FOLLOW_LIMITS}
    visited = {record['source_url'] for record in start_pages if record.get('source_url')}
    queued = set(visited)
    counts = {domain:0 for domain in FOLLOW_LIMITS}
    attempts = {domain:0 for domain in FOLLOW_LIMITS}
    crawl_log = []
    collected = []  # liste locale : la fonction ne dépend plus d'une variable globale

    def enqueue(record):
        if record.get('crawl_depth', 0) >= MAX_DEPTH:
            return
        for candidate in discover_document_links(record):
            if candidate['url'] not in queued and candidate['url'] not in visited:
                queues[candidate['domain']].append(candidate)
                queued.add(candidate['url'])

    for record in start_pages:
        record.setdefault('crawl_depth', 0)
        enqueue(record)

    if not EXPLORE_LINKS:
        print('Exploration désactivée ; seules les pages initiales sont utilisées.')
        return [], crawl_log, counts

    # Traitement par tours de rôle : aucun domaine ne monopolise la collecte.
    while any(queues[d] and counts[d] < FOLLOW_LIMITS[d] and attempts[d] < ATTEMPT_LIMITS[d]
              for d in FOLLOW_LIMITS):
        for domain in FOLLOW_LIMITS:
            if not queues[domain] or counts[domain] >= FOLLOW_LIMITS[domain] or attempts[domain] >= ATTEMPT_LIMITS[domain]:
                continue
            candidate = queues[domain].popleft()
            url = candidate['url']
            if url in visited:
                continue
            visited.add(url)
            attempts[domain] += 1
            html, status = get_html_guarded(url)
            crawl_log.append({**candidate, **status})
            if html:
                entry = {**candidate, 'html':html, 'origin':'live', 'source_url':url}
                collected.append(entry)
                counts[domain] += 1
                filename = hashlib.sha256(url.encode()).hexdigest()[:16] + '.html'
                (RAW_DIR / filename).write_text(html, encoding='utf-8')
                enqueue(entry)
            # On limite également le débit en présence d'erreurs HTTP.
            if status['status'] != 'skipped_robots':
                time.sleep(PAUSE_SECONDS)
            print(f"{domain:10s} | {status['status']:21s} | {counts[domain]:2d}/{FOLLOW_LIMITS[domain]:2d} | {url[-80:]}")
    return collected, crawl_log, counts

collected = []
if raw_live:
    # Audit préliminaire ; utile pour comprendre les rubriques repliées.
    summary = []
    for record in raw_live:
        links = discover_document_links(record)
        summary.append({'source': record['source_url'], 'liens_pertinents':len(links)})
    display(pd.DataFrame(summary))
    extra_pages, additional_log, domain_counts = crawl_controlled(raw_live)
    raw_live.extend(extra_pages)
    log.extend(additional_log)
    print('Pages supplémentaires récupérées :', len(extra_pages))
    print('Répartition :', domain_counts)
    print('Total de pages brutes pour ce corpus :', len(raw_live))
else:
    print('Aucune page officielle récupérée : exploration impossible ; corpus fictif conservé.')


### Exercice — Contrôler la couverture avant l'indexation

1. Vérifier combien de **mentions** et de **parcours** sont devenus des documents indépendants.
2. Identifier une fiche de master accessible depuis le catalogue mais absente du corpus (effet du plafond).
3. Comparer trois pages d'admission pour vérifier si elles comportent une année universitaire.
4. Expliquer pourquoi les liens vers les laboratoires hébergés sur d'autres domaines ne sont pas suivis automatiquement.
5. Justifier votre choix de plafonds et de profondeur : il s'agit d'un **échantillon**, pas d'un inventaire exhaustif.

> **Attention :** si l'accès direct renvoie une redirection ou une erreur réseau, l'exploration la consigne et passe à la suite. Le programme ne contourne ni accès restreint ni règle d'exploration.

## Étape 3 — Mode pédagogique de secours : six fiches **fictives**

Si la connexion à l'UPPA échoue (robots inaccessible, filtrage ou absence de réseau), le notebook reste entièrement exécutable sur des **exemples inventés**. Ils simulent des structures utiles (titres, listes, tableau, liens), **mais ne sont pas des sources de l’UPPA** et ne doivent jamais être cités comme telles.

**Règle :** le corpus final aura `dataset_kind='live'` ou `dataset_kind='synthetic_demo'` : les deux statuts ne sont jamais mélangés.

In [ ]:
# Une petite maquette contrôlée : deux documents fictifs par domaine.
FIXTURE_RECORDS = [
 ('formation','Fiche fictive — Master informatique',"""<h1>Master informatique — exemple fictif</h1><h2>Présentation</h2><p>Formation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle.</p><h2>Parcours</h2><ul><li>Industry 4.0 — cas fictif</li><li>Logistique et services — cas fictif</li></ul><h2>Compétences</h2><p>Concevoir des applications et analyser des données.</p>"""),
 ('formation','Parcours fictif — Industry 4.0',"""<h1>Parcours Industry 4.0 — exemple fictif</h1><h2>Objectifs</h2><p>Explorer la maintenance prédictive, les capteurs et le pilotage de production.</p><h2>Tableau des unités</h2><table><thead><tr><th>UE</th><th>Thème</th></tr></thead><tbody><tr><td>UE1</td><td>Capteurs</td></tr><tr><td>UE2</td><td>Données industrielles</td></tr></tbody></table>"""),
 ('admission','Procédure fictive — Master',"""<h1>Admission en master — exemple fictif</h1><h2>Documents</h2><ul><li>Curriculum vitae</li><li>Relevés de notes</li></ul><h2>Procédure</h2><p>Consulter les modalités officielles, différentes selon le profil et l'année.</p><p>Les dates citées dans ce document sont fictives.</p>"""),
 ('admission','Calendrier fictif — Master',"""<h1>Calendrier d'admission — exemple fictif</h1><h2>Dates illustratives</h2><table><tr><th>Phase</th><th>Date fictive</th></tr><tr><td>Candidature</td><td>15 janvier 2030</td></tr><tr><td>Réponse</td><td>20 mars 2030</td></tr></table><p>Ces dates ne concernent aucune campagne officielle.</p>"""),
 ('recherche','Laboratoire fictif — Systèmes numériques',"""<h1>Laboratoire Systèmes numériques — exemple fictif</h1><h2>Axes de recherche</h2><p>Réseaux, intelligence artificielle, simulation et systèmes cyber-physiques.</p><h2>Équipes</h2><p>Equipe Méthodes et Données.</p>"""),
 ('recherche','Projet fictif — Logistique intelligente',"""<h1>Projet Logistique intelligente — exemple fictif</h1><h2>Thématiques</h2><ul><li>Traçabilité</li><li>Optimisation d'entrepôt</li><li>Recommandation de services</li></ul><h2>Partenariats</h2><p>Exemple pédagogique sans partenaires réels.</p>""")
]

def html_fixture(title, body):
    return '<!doctype html><html lang="fr"><head><meta charset="utf-8"><title>'+title+'</title></head><body>' +         '<header><nav>MENU INTERDIT — ACCUEIL — RUBRIQUES</nav></header>' +         '<main><article>'+body+'</article></main><footer>PIED DE PAGE INTERDIT</footer></body></html>'

raw_demo = [
    {'domain':domain,'type':'fiction','url':None,'source_url':None,'origin':'synthetic_demo',
     'fixture_id':f'fiction_{i:02d}', 'html':html_fixture(title, body)}
    for i,(domain,title,body) in enumerate(FIXTURE_RECORDS,1)
]

RAW = raw_live if raw_live else raw_demo
DATASET_KIND = 'live' if raw_live else 'synthetic_demo'
print('Corpus retenu :', DATASET_KIND, '| documents :',len(RAW))
if DATASET_KIND != 'live':
    print('⚠️ Aucun document officiel n’a été téléchargé. Les exemples sont fictifs, à usage de test uniquement.')

## Étape 4 — Comprendre l'extraction de structure

Le problème central n'est pas de récupérer **tous les caractères**. Nous devons préserver :

| HTML source | Valeur pour un RAG | Format préparé |
|---|---|---|
| `<h1>`, `<h2>` | Titre et section | Titres Markdown `#`, `##` |
| `<p>` | Description, conditions, objectifs | Paragraphes |
| `<ul>`, `<li>` | Prérequis, pièces à fournir, compétences | Listes Markdown |
| `<table>` | Dates, maquettes, critères | Tableau Markdown |
| `<a href>` | Lien vérifiable | URL absolue |

La suppression des menus et pieds de page est faite **avant** la conversion Markdown. L’extracteur utilise `main` ou `article` lorsqu'ils existent, puis retire les blocs sans valeur documentaire. Les heuristiques ne sont pas universelles : il faut inspecter les résultats sur de vraies pages.

In [ ]:
def clean_markdown(text):
    """Nettoyage prudent : conserver titres, listes et tableaux."""
    text = text.replace('\r\n', '\n')
    text = re.sub(r'\n[ \t]+\n', '\n\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# Sélecteurs ciblés sur les éléments d'interface. Ne supprimer ni les
# tableaux, ni les listes de cours, ni les titres pédagogiques.
CHROME_SELECTORS = (
    'nav, footer, header, aside, script, style, noscript, form, button, '
    'input, select, iframe, svg, [aria-hidden="true"], [role="navigation"], '
    '.breadcrumb, .breadcrumbs, .breadcrumb-wrapper, .ariane, .fil-ariane, '
    '[id*="breadcrumb"], [class*="breadcrumb"], [class*="fil-ariane"], '
    '[aria-label="Breadcrumb"], [aria-label="Fil d’Ariane"], '
    '.share, .social-share, .page-tools, .print-page'
)


def remove_leading_chrome(md_text, title):
    """Retirer uniquement les lignes d'interface au DÉBUT d'un contenu.

    Une rubrique de navigation « Vous êtes ici » n'est pas une preuve documentaire.
    Ne jamais supprimer une ligne semblable rencontrée au milieu d'un article.
    """
    lines = md_text.splitlines()
    prefix = lines[:12]
    cursor = 0
    removable = {
        'vous êtes ici :', 'vous etes ici :', 'accueil', 'formations',
        'formation', 'admission', 'laboratoires', 'recherche'
    }
    while cursor < len(prefix):
        line=lines[cursor].strip()
        if not line or line.casefold() in removable:
            cursor += 1
            continue
        # Fil d'Ariane sous forme de liste en tête de fiche, seulement si
        # il précède le premier titre ou une zone d'information utile.
        if re.match(r'^\*\s+\[(?:Formations|Master|Admission|Recherche|Accueil)\]\(', line, re.I):
            cursor += 1
            continue
        break
    return clean_markdown('\n'.join(lines[cursor:]))


def classify_document_role(record, text_only, headings, links):
    """Conserver les pages hub dans le corpus mais permettre au NB02 de les écarter."""
    path=urlparse(record.get('source_url') or '').path.rstrip('/')
    known_hubs={'/fr/catalogue/toutes-les-formations.html',
                '/fr/trouver-votre-formation.html',
                '/fr/admission.html',
                '/fr/laboratoires.html'}
    if path in known_hubs:
        return 'navigation_hub'
    return 'content_page'


def extract_document(record):
    html = record['html']
    source_url = record.get('source_url')
    soup = BeautifulSoup(html, 'lxml')
    lang = soup.html.get('lang','') if soup.html else ''
    title_tag = soup.select_one('main h1, article h1, h1')
    document_title = (title_tag.get_text(' ',strip=True) if title_tag else
                      (soup.title.get_text(' ',strip=True) if soup.title else 'Sans titre'))
    main = soup.select_one('main, [role="main"], article, #contenu, #content')
    if main is None: main = soup.body or soup
    content = BeautifulSoup(str(main), 'lxml')
    for useless in list(content.select(CHROME_SELECTORS)):
        useless.decompose()
    for anchor in content.select('a[href]'):
        href=anchor.get('href','').strip()
        if href.lower().startswith(('javascript:', 'mailto:')):
            anchor.unwrap()  # texte conservé, URL dangereuse/parasite éliminée
        elif source_url:
            anchor['href'] = urljoin(source_url, href)
    markdown = html_to_markdown(str(content), heading_style='ATX', strip=['img']).strip()
    markdown = remove_leading_chrome(clean_markdown(markdown),document_title)
    # Le texte indexé doit correspondre au Markdown réellement exporté,
    # sans réintroduire le fil d'Ariane supprimé.
    text_only = re.sub(r'\s+', ' ', BeautifulSoup(
        markdown, 'html.parser').get_text(' ', strip=True)).strip()
    headings = [{'level':int(h.name[1]),'text':h.get_text(' ',strip=True)}
                for h in content.find_all(re.compile('^h[1-6]$'))]
    original_links=[]
    for a in content.select('a[href]'):
        target = urljoin(source_url, a['href']) if source_url else a['href']
        if target.startswith('http') and target not in original_links:
            original_links.append(target)
    # Année universitaire = deux années CONSÉCUTIVES (2026-2027). Un intervalle comme
    # « 2021-2026 » (contrat, projet) n'en est pas une. Forme normalisée : « AAAA-AAAA ».
    years=sorted({f'{a}-{b}' for a,b in re.findall(
        r'(?<!\d)(20\d{2})\s*[-–/]\s*(20\d{2})(?!\d)', markdown) if int(b)==int(a)+1})
    normal = re.sub(r'\s+',' ', markdown).casefold().strip()
    digest=hashlib.sha256(normal.encode('utf-8')).hexdigest()
    docid=hashlib.sha256((source_url or record.get('fixture_id','fiction')).encode()).hexdigest()[:16]
    role=classify_document_role(record,text_only,headings,original_links)
    return {
      'document_id':'uppa_'+docid if source_url else 'demo_'+docid,
      'domain':record['domain'], 'document_type':record['type'],
      'document_role':role,
      'discovered_from':record.get('discovered_from'),
      'link_label':record.get('link_label'), 'crawl_depth':record.get('crawl_depth',0),
      'title':document_title, 'source_url':source_url,
      'collected_at':datetime.now(timezone.utc).isoformat(timespec='seconds'),
      'dataset_kind':record['origin'], 'language':lang or None,
      'academic_years_detected':years, 'headings':headings, 'links':original_links,
      'table_count':len(content.find_all('table')),
      'content_markdown':markdown, 'content_text':text_only, 'sha256_text':digest,
      'character_count':len(text_only), 'word_count':len(text_only.split())
    }

# Tests de régression : les tableaux survivent, les menus et fils d'Ariane disparaissent.
t = extract_document(raw_demo[1])
assert '| UE1 | Capteurs |' in t['content_markdown']
assert 'MENU INTERDIT' not in t['content_markdown']
assert 'PIED DE PAGE INTERDIT' not in t['content_markdown']
assert 'Vous êtes ici :' not in remove_leading_chrome('Formations\n\nVous êtes ici :\n\n## Contenu', 'Contenu')
assert t['source_url'] is None and t['document_role']=='content_page'
print('Tests de nettoyage et de conservation des tableaux : OK')

In [ ]:
DOCS = [extract_document(item) for item in RAW]
print('Documents transformés :',len(DOCS))
for d in DOCS[:3]:
    print('•', d['domain'], '|', d['title'], '|', d['character_count'], 'caractères |', d['table_count'], 'tableaux')

display(Markdown('### Aperçu d’un Markdown extrait'))
print(DOCS[0]['content_markdown'][:1800])

### Exercice 2 — Inspecter un document métier

Pour **Industry 4.0**, retrouver dans le texte d’un master les sections « parcours », « compétences » ou « organisation ». Pour **SIGLIS**, repérer des mentions de logistique, WMS, chaîne d’information ou systèmes d’information. Pour **Admissions**, vérifier qu’un tableau conserve les phases et leurs dates sans décalage.

> L'absence de ces éléments dans une page donnée n'est pas une erreur de Python : le document peut tout simplement ne pas les contenir. Les captures du site doivent être vérifiées manuellement.

In [ ]:
# Changer l'index pour lire un autre document.
DOCUMENT_INDEX = 0
chosen = DOCS[DOCUMENT_INDEX]
print('Domaine :',chosen['domain'])
print('Titre :',chosen['title'])
print('URL :',chosen['source_url'] or '(exemple fictif, pas d’URL officielle)')
print('Sections :', [h['text'] for h in chosen['headings'][:12]])
display(Markdown(chosen['content_markdown'][:4000]))

## Étape 5 — Métadonnées et traçabilité

Le futur RAG doit citer **l’URL de la page** et, si possible, le titre de la section. Les informations d'admission sont souvent liées à un **millésime** : un calendrier 2026 ne doit pas être présenté comme valable pour 2027. Notre extracteur dégage uniquement les années **présentes dans le contenu**, sans deviner la période de validité.

**Dans le Notebook 02**, les métadonnées seront répétées sur chaque chunk. Une source sans URL (exemple fictif) ne pourra pas être présentée comme source officielle.

In [ ]:
for d in DOCS:
    print(f"{d['domain']:10s} | {d['title'][:50]:50s} | années = {d['academic_years_detected']}")
print('\nExemple de fiche JSON :')
preview = {k:DOCS[0][k] for k in ['document_id','domain','title','source_url','dataset_kind','collected_at','academic_years_detected','table_count']}
print(json.dumps(preview,indent=2,ensure_ascii=False))

## Étape 6 — Audit de la qualité

Une extraction peut être valide techniquement mais mauvaise scientifiquement : menu pris pour un article, tableau aplati, page pauvre, fiche en double, titre absent, métadonnées incomplètes.

Nous calculons des **indicateurs**, sans les appeler à tort « score de qualité absolu ». Les documents suspects seront soumis à une inspection humaine.

In [ ]:
quality_rows=[]
for d in DOCS:
    flagged=[]
    if d['character_count'] < 150: flagged.append('contenu_court')
    if not d['headings']: flagged.append('titres_absents')
    if d['domain']=='admission' and not d['academic_years_detected']: flagged.append('dates_a_verifier')
    if d['source_url'] is None: flagged.append('source_fictive')
    if d.get('document_role')=='navigation_hub': flagged.append('page_navigation_non_indexable')
    if re.search(r'(^|\n)Vous êtes ici\s*:', d['content_markdown'][:250],re.I):
        flagged.append('fil_ariane_residuel')
    quality_rows.append({
      'document_id':d['document_id'],'domaine':d['domain'],'titre':d['title'],
      'role':d.get('document_role','content_page'),
      'caracteres':d['character_count'],'mots':d['word_count'],
      'sections':len(d['headings']),'tableaux':d['table_count'],
      'liens':len(d['links']),'alertes':', '.join(flagged) or 'aucune alerte automatique'
    })
quality_df=pd.DataFrame(quality_rows)
display(quality_df)
counts=quality_df.groupby('domaine').agg(documents=('document_id','count'),mots=('mots','sum'),tableaux=('tableaux','sum'))
display(counts)

In [ ]:
# Visualiser la répartition documentaire, sans fixer les couleurs du graphique.
ax = quality_df.groupby('domaine')['mots'].sum().plot(kind='bar', figsize=(7,3.6))
ax.set_title('Volume textuel extrait par domaine (mots)')
ax.set_xlabel('Domaine'); ax.set_ylabel('Nombre de mots')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Doublons exacts sur contenu normalisé. Deux pages URL différentes peuvent avoir le même texte.
groups={}
for d in DOCS: groups.setdefault(d['sha256_text'], []).append(d)
duplicates=[g for g in groups.values() if len(g)>1]
print('Groupes de doublons textuels exacts :', len(duplicates))
for group in duplicates:
    print([x['title'] for x in group])
# La déduplication maintient une source unique et conserve l'inventaire complet séparément.
SEEN=set(); DOCS_UNIQUES=[]
for d in DOCS:
    if d['sha256_text'] not in SEEN:
        DOCS_UNIQUES.append(d); SEEN.add(d['sha256_text'])
print('Documents après suppression des doublons exacts :',len(DOCS_UNIQUES))

## Étape 6B — Tableau de bord de couverture de l’échantillon

Ce tableau compare le nombre de **liens candidats découverts**, les pages **tentées**, les pages **réussies**, les pages **refusées/ignorées** et les documents **exploitables pour le RAG**.

**Attention :** les candidats découverts ne correspondent pas au nombre total de formations de l’UPPA ; ils sont dédupliqués et limités aux règles de découverte du notebook. Une page « hub » peut être conservée comme source de liens, mais exclue des embeddings.

In [ ]:
# Reconstituer tous les candidats observables à partir des pages effectivement collectées.
# Cette opération est en mémoire : elle ne déclenche AUCUNE requête réseau.
DISCOVERED_LINKS=[]
for page in raw_live:
    DISCOVERED_LINKS.extend(discover_document_links(page))

found_per_domain={domain:set() for domain in ('formation','admission','recherche')}
for link in DISCOVERED_LINKS:
    found_per_domain[link['domain']].add(link['url'])

coverage=[]
for domain in ('formation','admission','recherche'):
    domain_log=[x for x in log if x.get('domain')==domain]
    attempts=[x for x in domain_log if x.get('url')]
    success=[x for x in attempts if x.get('status')=='ok']
    refusals=[x for x in attempts if x.get('status')!='ok']
    domain_docs=[d for d in DOCS_UNIQUES if d['domain']==domain]
    usable=[d for d in domain_docs if d.get('document_role')=='content_page'
            and d['character_count']>=150]
    coverage.append({
        'domaine':domain,
        # Les URL initiales sont aussi des candidates : sans elles, « tentées » dépasse « découverts ».
        'candidats_decouverts':len(found_per_domain[domain] | {x['url'] for x in attempts}),
        'urls_tentees':len({x['url'] for x in attempts}),
        'pages_telechargees':len({x['url'] for x in success}),
        'pages_refusees_ou_echouees':len(refusals),
        'documents_uniques':len(domain_docs),
        'pages_hub':sum(d.get('document_role')=='navigation_hub' for d in domain_docs),
        'documents_exploitables':len(usable),
    })
coverage_df=pd.DataFrame(coverage)
display(coverage_df)

fig,axes=plt.subplots(1,2,figsize=(12,3.8))
coverage_df.set_index('domaine')[['candidats_decouverts','urls_tentees','pages_telechargees']].plot.bar(ax=axes[0],rot=0)
axes[0].set_title('Couverture de l’échantillon exploré');axes[0].set_ylabel('Pages / URL')
coverage_df.set_index('domaine')[['documents_uniques','documents_exploitables']].plot.bar(ax=axes[1],rot=0)
axes[1].set_title('Documents collectés et réellement exploitables');axes[1].set_ylabel('Documents')
plt.tight_layout();plt.show()

status_df=pd.DataFrame(log)
if not status_df.empty:
    display(status_df.groupby(['domain','status'],as_index=False).size().rename(columns={'size':'nombre'}))
print('⚠️ Le tableau ne mesure PAS la couverture exhaustive du site UPPA.')

### Exercice 3 — Audit manuel (indispensable)

Sur au moins une page de chaque domaine, comparer l’URL originale avec le Markdown extrait :

| Critère | Questions à se poser |
|---|---|
| Fidélité du titre | Le nom exact du diplôme ou du laboratoire est-il préservé ? |
| Hiérarchie | Les titres de section correspondent-ils au contenu ? |
| Tableaux | Les lignes et colonnes d'admission restent-elles alignées ? |
| Liens | Peut-on retrouver la page ou l'annexe citée ? |
| Bruit | Restent-ils des menus, bandeaux, formulaires ou widgets ? |
| Temporalité | Les dates sont-elles accompagnées de leur année ou de leur contexte ? |

**Travail à remettre :** remplir une petite grille d'audit pour trois documents et décrire **deux défauts observés ou deux risques de défaut**. Le programme ne remplace pas cette vérification.

In [ ]:
manual_audit = pd.DataFrame([{
 'document_id':d['document_id'], 'domaine':d['domain'], 'titre':d['title'],
 'titres_ok':None,'tableaux_ok':None,'liens_ok':None,'bruit_residuel':None,
 'observation':''
} for d in DOCS_UNIQUES[:min(6,len(DOCS_UNIQUES))]])
display(manual_audit)
print('Compléter éventuellement le fichier CSV après export (la grille est volontairement non évaluée).')

## Étape 7 — Comparer BeautifulSoup et Crawl4AI (extension facultative)

**Objectif de la comparaison.** Exécuter **sur la même page autorisée** deux extracteurs différents et comparer : structure des titres, conservation des tableaux, des liens, taille du résultat et bruit résiduel. Le nombre de caractères n'est pas une preuve de meilleure qualité.

Crawl4AI requiert une installation distincte (bibliothèque Python, navigateur Playwright/Chromium). La documentation officielle : https://docs.crawl4ai.com/core/installation/ et https://docs.crawl4ai.com/api/crawl-result/.

**À activer uniquement si l'extraction HTTP initiale a été autorisée** et que l'environnement Colab dispose des dépendances nécessaires.

In [ ]:
# Option facultative. Modifiez la variable à True uniquement après avoir lu l'explication.
INSTALL_CRAWL4AI = False
if INSTALL_CRAWL4AI:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','crawl4ai'],check=True)
    subprocess.run(['crawl4ai-setup'], check=True)
    print('Crawl4AI installé. Tester la cellule suivante.')
else:
    print('Crawl4AI non installé : le parcours principal reste complet et fonctionnel.')

In [ ]:
RUN_CRAWL4AI_COMPARISON = False  # Activer après installation, uniquement avec pages live autorisées
async def crawl4ai_one_page(url):
    from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig
    # La permission robots.txt a déjà été vérifiée par robots_permission(url).
    # check_robots_txt renforce cette vérification côté navigateur.
    config = CrawlerRunConfig(check_robots_txt=True)
    async with AsyncWebCrawler(config=BrowserConfig(headless=True)) as crawler:
        result = await crawler.arun(url=url, config=config)
        if not result.success:
            raise RuntimeError(str(result.error_message))
        if not result.markdown:
            raise RuntimeError('Aucun Markdown retourné')
        md_result = result.markdown
        return md_result if isinstance(md_result,str) else (md_result.fit_markdown or md_result.raw_markdown)

if RUN_CRAWL4AI_COMPARISON and raw_live:
    target=raw_live[0]
    permitted, msg=robots_permission(target['source_url'])
    if permitted:
        try:
            crawl_markdown=await crawl4ai_one_page(target['source_url'])
            soup_markdown=extract_document(target)['content_markdown']
            comparison=pd.DataFrame([
                {'methode':'BeautifulSoup + Markdownify','caracteres':len(soup_markdown),
                 'titres':len(re.findall(r'^#{1,6} ',soup_markdown,re.M)),
                 'lignes_tableau':len(re.findall(r'^\|.*\|$',soup_markdown,re.M))},
                {'methode':'Crawl4AI','caracteres':len(crawl_markdown),
                 'titres':len(re.findall(r'^#{1,6} ',crawl_markdown,re.M)),
                 'lignes_tableau':len(re.findall(r'^\|.*\|$',crawl_markdown,re.M))}
            ])
            display(comparison)
            print('--- Aperçu Crawl4AI ---\n',crawl_markdown[:1000])
        except Exception as exc:
            print('Comparaison facultative indisponible :',str(exc)[:260])
    else:
        print('Comparaison annulée : accès non autorisé / non vérifiable.',msg)
else:
    print('Comparaison Crawl4AI désactivée : activer les deux options et disposer de pages live autorisées.')

## Étape 8 — Extension PDF (facultative)

Certains contenus, plaquettes et annexes sont diffusés au format PDF. Leur extraction peut être utile **après** vérification de l’URL, des conditions d'accès et du type MIME. Pour ces documents, utiliser `pymupdf` sur un fichier PDF **déjà obtenu légalement**, plutôt que de télécharger automatiquement tous les liens PDF découverts.

*Exercice :* choisir manuellement un PDF public autorisé, l'importer dans Colab, puis comparer l'ordre de lecture de deux pages. Les documents contenant des tableaux complexes pourront nécessiter une vérification humaine supplémentaire.

In [ ]:
# Extension manuelle (ne télécharge rien automatiquement).
# %pip -q install pymupdf
# import fitz
# pdf = fitz.open('/content/mon_pdf_autorise.pdf')
# texte = '\n'.join(page.get_text(sort=True) for page in pdf)
# print(texte[:1000])
print('Extraction PDF non déclenchée : import manuel et contrôle des droits requis.')

## Étape 9 — Export du corpus pour le Notebook 02

Les fichiers sont volontairement simples : **JSONL** pour les documents complets, **Markdown** pour la lecture humaine, **CSV** pour le contrôle rapide, **manifest JSONL** pour l’historique de collecte et **rapport CSV** de qualité.

**Convention importante :** les exports issus des exemples fictifs portent `synthetic_demo` dans leur nom et dans leurs enregistrements. **Ne pas les utiliser pour faire des affirmations sur les formations, admissions ou laboratoires réels.**

In [ ]:
# Sécuriser les noms et identifier sans ambiguïté les corpus fictifs.
suffix = 'uppa' if DATASET_KIND == 'live' else 'synthetic_demo'
corpus_file = EXPORT_DIR / f'corpus_{suffix}.jsonl'
manifest_file = EXPORT_DIR / f'manifest_{suffix}.jsonl'

with corpus_file.open('w',encoding='utf-8') as f:
    for d in DOCS_UNIQUES:
        f.write(json.dumps(d,ensure_ascii=False)+'\n')
with manifest_file.open('w',encoding='utf-8') as f:
    audit_log = log + ([{'domain':d['domain'], 'url':None, 'status':'synthetic_demo',
                        'detail':'Document fictif à des fins pédagogiques', 'fixture_id':d['fixture_id']}
                       for d in raw_demo] if DATASET_KIND == 'synthetic_demo' else [])
    for record in audit_log:
        f.write(json.dumps(record,ensure_ascii=False)+'\n')

for d in DOCS_UNIQUES:
    filename=d['document_id']+'.md'
    (PROCESSED_DIR / filename).write_text(d['content_markdown'],encoding='utf-8')
quality_df.to_csv(EXPORT_DIR / f'rapport_qualite_{suffix}.csv',index=False,encoding='utf-8-sig')
manual_audit.to_csv(EXPORT_DIR / f'audit_manuel_a_completer_{suffix}.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(SEEDS).to_csv(EXPORT_DIR / 'sources_initiales.csv',index=False,encoding='utf-8-sig')

manifest_summary={
 'created_at':datetime.now(timezone.utc).isoformat(timespec='seconds'),
 'dataset_kind':DATASET_KIND,'n_documents':len(DOCS_UNIQUES),
 'domains':dict(Counter(x['domain'] for x in DOCS_UNIQUES)),
 'schema_version':'uppa_rag_doc_v1','requires_verification':True,
 'official_sources_collected':DATASET_KIND=='live',
 'note':'Données pédagogiques fictives' if DATASET_KIND!='live' else 'Extraits de pages publiques accessibles lors de la collecte'
}
(EXPORT_DIR/'dataset_info.json').write_text(json.dumps(manifest_summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Fichiers principaux :')
for p in [corpus_file, manifest_file, EXPORT_DIR/'dataset_info.json']:
    print('-',p, f'({p.stat().st_size} octets)')

# Le dashboard est un livrable autonome réutilisable pour le rapport M2.
coverage_df.to_csv(EXPORT_DIR/f'couverture_sources_{suffix}.csv',index=False,encoding='utf-8-sig')
status_df.to_csv(EXPORT_DIR/f'journal_statuts_{suffix}.csv',index=False,encoding='utf-8-sig')


In [ ]:
# Validation avant passage au Notebook 02.
loaded = [json.loads(line) for line in corpus_file.read_text(encoding='utf-8').splitlines() if line.strip()]
assert loaded and all('content_markdown' in d and 'domain' in d for d in loaded)
assert all(d['dataset_kind'] == DATASET_KIND for d in loaded)
assert len({d['document_id'] for d in loaded}) == len(loaded)
assert all((d['source_url'] is not None) == (DATASET_KIND=='live') for d in loaded)
print('✅ Tests du corpus exporté : OK')
print('Domaines représentés :',dict(Counter(d['domain'] for d in loaded)))
if DATASET_KIND!='live':
    print('⚠️ CORPUS FICTIF — aucun fait de ce corpus ne doit être attribué à l’UPPA.')

In [ ]:
# ZIP comprenant corpus + journal + contrôle qualité + fichiers Markdown + les snapshots HTML.
archive=BASE / f'UPPA_Notebook01_exports_{suffix}.zip'
current_processed = [PROCESSED_DIR / (d['document_id'] + '.md') for d in DOCS_UNIQUES]
current_export = [corpus_file, manifest_file, EXPORT_DIR/'dataset_info.json',
                  EXPORT_DIR/f'rapport_qualite_{suffix}.csv',
                  EXPORT_DIR/f'audit_manuel_a_completer_{suffix}.csv',
                  EXPORT_DIR/'sources_initiales.csv',
                  EXPORT_DIR/f'couverture_sources_{suffix}.csv',
                  EXPORT_DIR/f'journal_statuts_{suffix}.csv']
current_raw = [RAW_DIR/(hashlib.sha256(d['source_url'].encode()).hexdigest()[:16]+'.html')
               for d in DOCS_UNIQUES if d['source_url']]
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in current_processed + current_export + current_raw:
        if file.exists(): z.write(file,file.relative_to(BASE))
print('Archive créée :',archive, 'taille :',round(archive.stat().st_size/1024),'Ko')
try:
    from google.colab import files
    # Ne pas télécharger automatiquement sans action de l'étudiant.
    print('Pour télécharger dans Colab : files.download(str(archive))')
except ImportError:
    print('Le fichier ZIP est disponible dans le répertoire affiché.')

## Étape 10 — Interpréter et rendre le travail

### Questions de synthèse (à rédiger dans votre compte rendu)

1. Pourquoi conserver le HTML brut et le Markdown séparément ?
2. Quels éléments de l'interface Web ont pollué l'extraction ?
3. Une page de catalogue et une fiche de formation doivent-elles être indexées avec la même priorité ? Pourquoi ?
4. Quels risques introduisent les tableaux d'admission et leurs millésimes ?
5. Comment garantir que la source citée par le RAG correspond au bon établissement, au bon parcours et à la bonne campagne ?
6. BeautifulSoup et Crawl4AI produisent-ils les mêmes sections et tableaux ? Présenter des **preuves observables**, pas une intuition.
7. Quelles informations faudra-t-il ajouter aux documents avant la recherche vectorielle ?

### Résultat attendu
- Un ensemble de documents **réels et autorisés**, ou à défaut une démonstration **fictive clairement étiquetée**.
- Un manifeste d’acquisition et un rapport des erreurs/limitations.
- Un corpus structuré dont chaque document possède titre, domaine, contenu et provenance.
- Un audit humain de trois documents (une page par domaine si disponibles).
- Une archive pour le Notebook 02 : **Chunking, embeddings et Qdrant**.

### À retenir

> **Un RAG fiable commence par la qualité et la traçabilité de son corpus.** Une information d’admission erronée, un tableau mal reconstruit ou une source mal attribuée ne seront pas corrigés automatiquement par le modèle de langage.

**Références techniques :** [UPPA — Formations](https://formation.univ-pau.fr/fr/trouver-votre-formation.html) · [UPPA — Admissions](https://formation.univ-pau.fr/fr/admission.html) · [UPPA — Recherche](https://recherche.univ-pau.fr/fr/laboratoires.html) · [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) · [Crawl4AI](https://docs.crawl4ai.com/) · [Markdownify](https://github.com/matthewwithanm/python-markdownify).

### Analyse complémentaire de couverture (version 1.1)
Comparer le nombre de liens découverts et le nombre de pages effectivement téléchargées.
Décrire les limites de l’échantillon, y compris les erreurs HTTP, les plafonds et les URL externes.


### Résultats attendus — améliorer son corpus

Avant d’exécuter le Notebook 02, comparer les colonnes `candidats_decouverts`, `pages_telechargees` et `documents_exploitables`. Ne pas indexer automatiquement les pages d’interface (`document_role=navigation_hub`) ; elles servent surtout à trouver d’autres fiches. Vérifier manuellement les documents d’admission datés.